# MoE Training Notebook — Kaggle T4 GPU

Trains the Mixture-of-Experts components (C3 routing gate + C5 expert bank + C6 fusion)
in two phases using real CXR images from TBX11K and Montgomery datasets.

| Phase | Script | What trains | Time (T4) |
|-------|--------|-------------|----------|
| 0 | This notebook | MoE cache generation (embeddings + pseudo-masks) | ~30 min |
| 1 | `train_experts.py` | Each expert decoder independently | ~1 hr |
| 2 | `train_moe_joint.py` | Routing gate + experts + fusion end-to-end | ~2 hr |

Output: `/kaggle/working/moe_runs/moe_best.pt` — upload this as a new Kaggle dataset
and point `component-moe` slug to it in `final_s.ipynb`.

## 1. Clone Repository + Install Dependencies

In [ ]:
import os

REPO_DIR = '/kaggle/working/dl-project-codebase'
if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/mabdullahi7780/dl-project-codebase.git {REPO_DIR}')
os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
%%time
import subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'],
    capture_output=True, text=True
)
print(result.stdout[-1000:] if result.stdout else '(no output)')

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'git+https://github.com/facebookresearch/segment-anything.git',
     'safetensors'],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else '(no output)')
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])

import importlib.util
assert importlib.util.find_spec('segment_anything') is not None
print('segment_anything: OK')

## 2. Environment Variables + Checkpoint Symlinks

In [ ]:
import os, glob
from pathlib import Path

os.environ['MONTGOMERY_ROOT'] = '/kaggle/input/datasets/iahmedhabib/montgomery'
os.environ['SHENZHEN_ROOT']   = '/kaggle/input/datasets/iahmedhabib/shehzhenn'
os.environ['TBX11K_ROOT']     = '/kaggle/input/datasets/usmanshams/tbx-11/TBX11K'
os.environ['MEDSAM_VIT_B_CKPT'] = '/kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth'

for label, path in [
    ('MONTGOMERY_ROOT', os.environ['MONTGOMERY_ROOT']),
    ('TBX11K_ROOT',     os.environ['TBX11K_ROOT']),
]:
    p = Path(path)
    print(f'  {"[OK]" if p.exists() else "[MISS]"} {label} -> {path}')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nDevice: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# Symlink checkpoints the pipeline expects
CHECKPOINT_LINKS = [
    ('MedSAM ViT-B backbone',
     ['/kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth',
      '/kaggle/input/**/medsam_vit_b.pth'],
     'checkpoints/medsam/medsam_vit_b.pth'),
    ('Component 1 LoRA+DANN adapters',
     ['/kaggle/input/datasets/iahmedhabib/component1-artifacts/component1_adapters.safetensors',
      '/kaggle/input/**/component1_adapters.safetensors'],
     'checkpoints/component1/component1_adapters.safetensors'),
    ('Existing moe_best.pt (resume)',
     ['/kaggle/input/datasets/iahmedhabib/component-moe/moe_best.pt',
      '/kaggle/input/**/moe_best.pt'],
     'checkpoints/component_moe/moe_best.pt'),
]

def resolve_candidate(candidates):
    for c in candidates:
        if any(ch in c for ch in '*?['):
            for m in sorted(glob.glob(c, recursive=True)):
                if Path(m).is_file(): return Path(m)
        else:
            p = Path(c)
            if p.is_file(): return p
    return None

for label, candidates, dst in CHECKPOINT_LINKS:
    dst_p = Path(dst)
    dst_p.parent.mkdir(parents=True, exist_ok=True)
    src_p = resolve_candidate(candidates)
    if src_p is None:
        print(f'  [MISS] {label}')
        continue
    if dst_p.exists() or dst_p.is_symlink(): dst_p.unlink()
    os.symlink(src_p.resolve(), dst_p)
    print(f'  [OK]  {label} ({src_p.stat().st_size/1e6:.1f} MB)')

## 3. Override Training Config

We override the defaults in `moe.yaml` for a **full training run** (not gate-only):
- `gate_only: false` — train experts + fusion too, not just the gate
- More epochs for both phases
- `use_domain_ctx: true` — gate uses TXV domain embeddings

In [ ]:
import yaml
from pathlib import Path

cfg_path = Path('configs/moe.yaml')
cfg = yaml.safe_load(cfg_path.read_text())

# Enable domain context in routing gate
cfg['moe']['use_domain_ctx'] = True
cfg['moe']['routing_temperature'] = 0.5  # sharper routing

# Phase 1: expert pretraining
cfg.setdefault('moe_training', {}).setdefault('pretrain', {})
cfg['moe_training']['pretrain']['epochs'] = 25
cfg['moe_training']['pretrain']['batch_size'] = 8
cfg['moe_training']['pretrain']['lr'] = 1e-4

# Phase 2: joint training — full (not gate-only)
cfg['moe_training'].setdefault('joint', {})
cfg['moe_training']['joint']['epochs'] = 20
cfg['moe_training']['joint']['batch_size'] = 4
cfg['moe_training']['joint']['gate_only'] = False   # train EVERYTHING
cfg['moe_training']['joint']['lr_gate'] = 5e-4
cfg['moe_training']['joint']['lr_experts'] = 1e-4
cfg['moe_training']['joint']['lr_fusion'] = 1e-4
cfg['moe_training']['joint']['load_balance_weight'] = 0.05
cfg['moe_training']['joint']['expert_aux_weight'] = 0.3
cfg['moe_training']['joint']['resume_from_moe_best'] = None  # fresh start

cfg['moe_training']['save_dir'] = '/kaggle/working/moe_runs'
cfg['moe_training']['amp'] = True
cfg['moe_training']['num_workers'] = 2

# Write patched config
train_cfg_path = Path('/kaggle/working/moe_train.yaml')
train_cfg_path.write_text(yaml.dump(cfg))
print('Training config written to:', train_cfg_path)
print(f'  Phase 1 epochs: {cfg["moe_training"]["pretrain"]["epochs"]}')
print(f'  Phase 2 epochs: {cfg["moe_training"]["joint"]["epochs"]}')
print(f'  gate_only: {cfg["moe_training"]["joint"]["gate_only"]}')
print(f'  use_domain_ctx: {cfg["moe"]["use_domain_ctx"]}')

## 4. Generate MoE Cache

Runs Component 1 (encoder) + Component 2 (TXV) on all training images and saves
per-image `.pt` cache files containing:
- `image_emb` — [256, 64, 64] from Component 1 backbone
- `domain_ctx` — [256] from Component 2 routing head
- `lesion_mask` — [1, 256, 256] pseudo-mask (Grad-CAM from TXV TB-mimic classes)
- `expert_masks` — dict of 4 expert pseudo-masks from pathology-specific Grad-CAM
- `supervision_weight` — 1.0 for TB+, 0.5 for TB-

Expert ↔ TXV class mapping:
| Expert | Pathology | TXV class |
|--------|-----------|----------|
| consolidation | Dense opacity | consolidation (idx 1) |
| cavity | Ring lesion | lung_lesion (idx 14) + mass (idx 12) |
| fibrosis | Reticular | fibrosis (idx 6) + infiltration (idx 2) |
| nodule | Focal opacity | nodule (idx 11) |

In [ ]:
%%time
import torch
import torch.nn.functional as F
import numpy as np
from pathlib import Path

from src.app.infer import build_models, load_baseline_config
from src.components.component0_qc import harmonise_sample
from src.core.device import pick_device

import yaml
cfg = yaml.safe_load(Path('/kaggle/working/moe_train.yaml').read_text())
dev = pick_device(cfg.get('runtime', {}).get('device'))

print('Loading Component 1 + Component 2 (frozen, for embedding generation)...')
c1, c2, c4 = build_models(cfg, dev)
print(f'  C1 backend: {c1.encoder.active_backend}')
print(f'  C2 backend: {c2.active_backend}')

# TXV class indices for expert pseudo-masks
EXPERT_CLASS_INDICES = {
    'consolidation': [1],           # consolidation
    'cavity':        [14, 12],      # lung_lesion, mass
    'fibrosis':      [6, 2],        # fibrosis, infiltration
    'nodule':        [11],          # nodule
}
from src.components.component5_experts import EXPERT_NAMES
print(f'  Expert order: {EXPERT_NAMES}')


def make_gradcam_mask(features_7x7, classifier_weight, class_indices, size=256):
    """Grad-CAM-style class activation map → binary pseudo-mask [1,size,size]."""
    # features_7x7: [1, 1024, 7, 7]
    # classifier_weight: [n_classes, 1024]
    cam = torch.zeros(7, 7, device=features_7x7.device)
    for idx in class_indices:
        w = classifier_weight[idx]          # [1024]
        # weighted sum over channels
        cam += (w.view(1024, 1, 1) * features_7x7[0]).sum(dim=0)  # [7,7]
    cam = F.relu(cam)
    # Normalise to [0,1]
    cam_min, cam_max = cam.min(), cam.max()
    if cam_max > cam_min:
        cam = (cam - cam_min) / (cam_max - cam_min + 1e-8)
    # Upsample to [1, size, size]
    cam_up = F.interpolate(cam.unsqueeze(0).unsqueeze(0), size=(size, size),
                           mode='bilinear', align_corners=False)[0]  # [1,size,size]
    return cam_up


def process_image(img_path, dataset_id, tb_label):
    """Generate one cache entry. Returns dict or None on error."""
    from PIL import Image as PILImage
    import numpy as np
    try:
        with PILImage.open(img_path) as pil:
            arr = np.asarray(pil.convert('L'))
        sample = {'image': arr, 'image_id': Path(img_path).stem,
                  'dataset_id': dataset_id, 'view': None,
                  'pixel_spacing_cm': None, 'path': str(img_path)}
        harmonised = harmonise_sample(sample)

        x_1024 = harmonised.x_1024.unsqueeze(0).to(dev)   # [1,1,1024,1024]
        x_3ch  = harmonised.x_3ch.unsqueeze(0).to(dev)    # [1,3,1024,1024]
        x_224  = harmonised.x_224.unsqueeze(0).to(dev)    # [1,1,224,224]

        with torch.no_grad():
            # Component 1 → image embedding
            img_emb, _ = c1(x_3ch, torch.zeros(1, dtype=torch.long, device=dev))
            # img_emb: [1, 256, 64, 64]

            # Component 2 → domain context + features for Grad-CAM
            txv_out = c2.forward_features(x_224)
            features_7x7   = txv_out.features_7x7       # [1,1024,7,7]
            domain_ctx     = txv_out.domain_ctx[0]       # [256]
            classifier_w   = txv_out.classifier_weight   # [n_classes, 1024] or None

        # Build expert pseudo-masks
        expert_masks = {}
        if classifier_w is not None:
            for exp_name in EXPERT_NAMES:
                idxs = EXPERT_CLASS_INDICES.get(exp_name, [1])
                cam = make_gradcam_mask(features_7x7, classifier_w, idxs)
                # Only activate for TB+ images; zero mask for TB-
                expert_masks[exp_name] = cam.cpu() if tb_label == 1 else torch.zeros_like(cam)
        else:
            # Mock backend — use random blobs for TB+, zeros for TB-
            for exp_name in EXPERT_NAMES:
                if tb_label == 1:
                    mask = torch.zeros(1, 256, 256)
                    cx, cy = np.random.randint(64, 192, 2)
                    rx, ry = np.random.randint(20, 60, 2)
                    yy, xx = torch.meshgrid(torch.arange(256), torch.arange(256), indexing='ij')
                    blob = ((xx - cx).float()/rx)**2 + ((yy - cy).float()/ry)**2
                    mask[0] = (blob < 1.0).float()
                    expert_masks[exp_name] = mask
                else:
                    expert_masks[exp_name] = torch.zeros(1, 256, 256)

        # Fused lesion mask = max across experts
        if expert_masks:
            lesion_mask = torch.stack(list(expert_masks.values())).max(dim=0).values
        else:
            lesion_mask = torch.zeros(1, 256, 256)

        return {
            'image_emb':   img_emb[0].cpu(),       # [256,64,64]
            'domain_ctx':  domain_ctx.cpu(),        # [256]
            'lesion_mask': lesion_mask.cpu(),       # [1,256,256]
            'expert_masks': {k: v.cpu() for k, v in expert_masks.items()},
            'supervision_weight': 1.0 if tb_label == 1 else 0.5,
            'dataset_id': dataset_id,
            'tb_label': tb_label,
        }
    except Exception as exc:
        print(f'  ERROR {Path(img_path).name}: {exc}')
        return None

print('Component 1 + 2 loaded. Ready for cache generation.')

In [ ]:
%%time
import time
from pathlib import Path

CACHE_DIR = Path('/kaggle/working/moe_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

montgomery_root = Path(os.environ['MONTGOMERY_ROOT'])
tbx11k_root     = Path(os.environ['TBX11K_ROOT'])

# Collect training images with labels
# Montgomery: suffix _0 = TB-, _1 = TB+
# TBX11K:     folder tb/ or sick/ = TB+, health/ = TB-
training_images = []  # list of (path, dataset_id, tb_label)

# Montgomery
if montgomery_root.exists():
    imgs_dir = montgomery_root / 'CXR_png'
    if not imgs_dir.exists():
        imgs_dir = montgomery_root
    for f in sorted(imgs_dir.rglob('*.png')) + sorted(imgs_dir.rglob('*.jpg')):
        stem = f.stem
        if stem.endswith('_1'):   lbl = 1
        elif stem.endswith('_0'): lbl = 0
        else: continue
        training_images.append((f, 'montgomery', lbl))
    print(f'Montgomery: {len(training_images)} images')

# TBX11K
tbx_count_before = len(training_images)
if tbx11k_root.exists():
    imgs_root = tbx11k_root / 'imgs'
    if not imgs_root.exists(): imgs_root = tbx11k_root
    for folder, lbl in [('tb', 1), ('sick', 1), ('health', 0)]:
        folder_path = imgs_root / folder
        if folder_path.exists():
            imgs = sorted(folder_path.rglob('*.png')) + sorted(folder_path.rglob('*.jpg'))
            for f in imgs:
                training_images.append((f, 'tbx11k', lbl))
    print(f'TBX11K: {len(training_images) - tbx_count_before} images')

# Limit per class to avoid huge cache (cap at 300 TB+ and 300 TB-)
import random
random.seed(42)
pos = [(p,d,l) for p,d,l in training_images if l==1]
neg = [(p,d,l) for p,d,l in training_images if l==0]
random.shuffle(pos); random.shuffle(neg)
MAX_PER_CLASS = 300
training_images = pos[:MAX_PER_CLASS] + neg[:MAX_PER_CLASS]
random.shuffle(training_images)

print(f'\nTotal training images: {len(training_images)}  '
      f'(TB+: {sum(l==1 for _,_,l in training_images)}  '
      f'TB-: {sum(l==0 for _,_,l in training_images)})')

# Generate cache
n_done, n_err = 0, 0
t0 = time.time()
for img_path, dataset_id, tb_label in training_images:
    out_path = CACHE_DIR / f'{dataset_id}_{Path(img_path).stem}.pt'
    if out_path.exists():
        n_done += 1
        continue
    result = process_image(img_path, dataset_id, tb_label)
    if result is not None:
        torch.save(result, out_path)
        n_done += 1
    else:
        n_err += 1
    if (n_done + n_err) % 50 == 0:
        elapsed = time.time() - t0
        print(f'  {n_done+n_err}/{len(training_images)}  '
              f'({(n_done+n_err)/elapsed:.2f} img/s)  errors={n_err}')

cache_files = sorted(CACHE_DIR.glob('*.pt'))
print(f'\nCache complete: {len(cache_files)} files in {CACHE_DIR}  '
      f'(errors: {n_err})  '
      f'time: {time.time()-t0:.1f}s')

## 5. Phase 1 — Expert Pretraining

Each of the 4 expert decoders is pretrained independently on its pathology
using the Grad-CAM pseudo-masks generated above.

In [ ]:
%%time
import yaml
from pathlib import Path
from src.training.train_experts import train_single_expert
from src.components.component5_experts import EXPERT_NAMES

cfg = yaml.safe_load(Path('/kaggle/working/moe_train.yaml').read_text())
cache_dir = Path('/kaggle/working/moe_cache')
save_dir  = Path('/kaggle/working/moe_runs')
cfg['moe_training']['save_dir'] = str(save_dir)

print(f'Phase 1: training {len(EXPERT_NAMES)} experts × '
      f'{cfg["moe_training"]["pretrain"]["epochs"]} epochs each')
print(f'Cache: {len(list(cache_dir.glob("*.pt")))} samples\n')

for expert_name in EXPERT_NAMES:
    print(f'=== Expert: {expert_name} ===')
    train_single_expert(expert_name, cfg, cache_dir=cache_dir)
    print()

expert_ckpts = sorted(save_dir.glob('expert_*_best.pt'))
print(f'Phase 1 complete. Saved {len(expert_ckpts)} expert checkpoints:')
for p in expert_ckpts:
    print(f'  {p.name}  ({p.stat().st_size/1e6:.2f} MB)')

## 6. Phase 2 — Joint MoE Training

Trains routing gate (C3) + all 4 expert decoders (C5) + fusion module (C6)
end-to-end. Experts are warm-started from Phase 1 checkpoints.

In [ ]:
%%time
import yaml
from pathlib import Path
from src.training.train_moe_joint import train_joint

cfg = yaml.safe_load(Path('/kaggle/working/moe_train.yaml').read_text())
cfg['moe_training']['save_dir'] = '/kaggle/working/moe_runs'

cache_dir = Path('/kaggle/working/moe_cache')

print(f'Phase 2: joint training')
print(f'  epochs: {cfg["moe_training"]["joint"]["epochs"]}')
print(f'  gate_only: {cfg["moe_training"]["joint"]["gate_only"]}')
print(f'  load_balance_weight: {cfg["moe_training"]["joint"]["load_balance_weight"]}')
print(f'  expert_aux_weight: {cfg["moe_training"]["joint"]["expert_aux_weight"]}')
print()

final_path = train_joint(cfg, cache_dir=cache_dir)

save_dir = Path('/kaggle/working/moe_runs')
moe_best = save_dir / 'moe_best.pt'
print(f'\nPhase 2 complete.')
print(f'  moe_best.pt: {moe_best}  ({moe_best.stat().st_size/1e6:.2f} MB)')

## 7. Validate Trained Checkpoint

Quick sanity check — run one inference with the freshly trained `moe_best.pt`
and confirm the MoE path fires correctly.

In [ ]:
%%time
import tempfile, json
import numpy as np
from pathlib import Path
from PIL import Image
import yaml

from src.app.infer import run_single_image_inference

# Point moe config at the newly trained checkpoint
cfg = yaml.safe_load(Path('/kaggle/working/moe_train.yaml').read_text())
cfg['moe']['checkpoint_path'] = '/kaggle/working/moe_runs/moe_best.pt'

val_cfg_path = Path('/kaggle/working/moe_val.yaml')
val_cfg_path.write_text(yaml.dump(cfg))

with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir = Path(tmpdir)
    arr = np.random.randint(0, 256, (1024, 1024), dtype=np.uint8)
    img_path = tmpdir / 'val.png'
    Image.fromarray(arr, mode='L').save(img_path)

    bundle = run_single_image_inference(
        image_path=img_path,
        dataset='montgomery',
        outdir=tmpdir / 'out',
        config_path=val_cfg_path,
    )

assert bundle.pipeline_mode == 'moe', f'Expected moe, got {bundle.pipeline_mode}'
assert bundle.routing_weights is not None
print(f'pipeline_mode:    {bundle.pipeline_mode}')
print(f'routing_weights:  {bundle.routing_weights.round(3)}')
print(f'ALP:              {bundle.evidence_json["scoring"]["ALP"]:.2f}')
print(f'cavity_flag:      {bundle.evidence_json["scoring"]["cavity_flag"]}')
print('Validation: OK')

## 8. Summary + Next Steps

The trained checkpoint is at `/kaggle/working/moe_runs/moe_best.pt`.

**To use it in `final_s.ipynb`:**
1. Download `moe_best.pt` from Kaggle output
2. Upload it to your `component-moe` Kaggle dataset (replacing the old one)
3. Run `final_s.ipynb` — it will automatically pick up the new checkpoint
   via the symlink in §3b.

In [ ]:
from pathlib import Path
import json

save_dir = Path('/kaggle/working/moe_runs')

# Print training history summary
history_file = save_dir / 'moe_joint_history.jsonl'
if history_file.exists():
    history = [json.loads(l) for l in history_file.read_text().splitlines() if l.strip()]
    if history:
        best = min(history, key=lambda r: r['total'])
        last = history[-1]
        print('Joint training history:')
        print(f'  Best epoch {best["epoch"]}: total={best["total"]:.4f}  '
              f'mask={best["mask_loss"]:.4f}  lb={best["lb_loss"]:.4f}')
        print(f'  Last epoch {last["epoch"]}: total={last["total"]:.4f}  '
              f'mask={last["mask_loss"]:.4f}  lb={last["lb_loss"]:.4f}')

print('\nOutput files:')
for p in sorted(save_dir.glob('*.pt')):
    print(f'  {p.name:<35} {p.stat().st_size/1e6:.2f} MB')

moe_best = save_dir / 'moe_best.pt'
print(f'\nUpload this to Kaggle: {moe_best}')